# Chapter 1: Understanding LLM Inputs — Tokenization, Datasets, and Embeddings

This notebook covers the standard pipeline for preparing text data to be ingested by a Large Language Model (LLM):
1. **Tokenization**: Converting raw text into individual words, characters, or subwords.
2. **Vocabulary Construction**: Mapping unique tokens to unique integers (Token IDs).
3. **BytePair Encoding (BPE)**: Leveraging subword tokenizers like Tiktoken (used in GPT models).
4. **Sliding Window Data Sampling**: Formatting token IDs into input-target context pairs.
5. **Embeddings**: Converting Token IDs and their positions into multi-dimensional vectors.

## 1. Installation & Environment Setup

First, let's install the required packages: PyTorch for modeling, and Tiktoken for BytePair Encoding.

In [ ]:
# Install torch and tiktoken silently
!uv pip install torch tiktoken --quiet

In [ ]:
# Verify the installed versions
from importlib.metadata import version

print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

## 2. Load the Dataset

We will load Edith Wharton's short story *"The Verdict"* to use as our text corpus.

In [ ]:
# Read the text file
with open("./the-verdict.txt", "r", encoding='utf-8') as f:
    raw_text = f.read()

print("Total number of characters:", len(raw_text))
print("Preview first 100 characters:")
print(raw_text[:99])

## 3. Basic Tokenization

We begin by understanding basic tokenization by splitting text using regular expressions.

### 3.1 Splitting by Punctuation and Whitespace

In [ ]:
import re

text = "Hello, world. This, is a test."
# Splitting on commas, periods, and whitespace. 
# The capture parenthesis preserves the splitting characters in the output list.
result = re.split(r'([,.]|\s)', text)

print(result)

In [ ]:
# Clean up empty string artifacts and whitespaces
result = [item for item in result if item.strip()]

print(result)

### 3.2 Advanced Punctuation Regex Splitting

In [ ]:
# Handle a wider range of punctuation symbols and double dashes (--)
result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]

print(result)

### 3.3 Tokenizing the Full Corpus

In [ ]:
# Apply the advanced punctuation split to the entire text file
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]

print("First 30 tokens of preprocessed text:")
print(preprocessed[:30])
print("\nTotal tokens count:", len(preprocessed))

## 4. Converting Tokens into Token IDs

To feed tokens into an LLM, we map each unique word/token to a unique integer index. This is called the *vocabulary*.

### 4.1 Constructing the Vocabulary

In [ ]:
# Deduplicate tokens and sort them to get the set of all unique words
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)

print("Vocabulary Size:", vocab_size)

In [ ]:
# Create the token-to-integer dictionary mapping
vocab = {token:integer for integer, token in enumerate(all_words)}

In [ ]:
# View the first 50 entries in our vocabulary map
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

### 4.2 Building a Simple Tokenizer (Version 1)

Let's write a simple class containing an `encode` method (text -> integer IDs) and `decode` method (integer IDs -> text).

In [ ]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}
    
    def encode(self, text):
        # Perform the same preprocessing splitting
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        # Map strings to integers
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        # Map integers back to strings
        text = " ".join([self.int_to_str[i] for i in ids])
        # Remove leading spaces before punctuation symbols to restore readability
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [ ]:
# Instantiate and check the tokenizer on a sample text from the book
tokenizer = SimpleTokenizerV1(vocab)

text = """"It's the last he painted, you know," 
           Mrs. Gisburn said with pardonable pride."""

print("Sample Text:")
print(text)

In [ ]:
# Encode the sample text
ids = tokenizer.encode(text)
print("Token IDs:", ids)

In [ ]:
# Decode back to verify readability
decoded_text = tokenizer.decode(ids)
print("Decoded Text:", decoded_text)

#### The OOV (Out-of-Vocabulary) Problem

If we try to encode words that were not in the training corpus (like "Hello" or "tea"), the V1 tokenizer will raise a `KeyError`.

In [ ]:
tokenizer = SimpleTokenizerV1(vocab)
text_oov = "Hello, do you like tea. Is this-- a test?"

# Running the line below would error:
# tokenizer.encode(text_oov)

### 4.3 Handling OOV Tokens & Special Tokens (Version 2)

To solve this, we add special tokens like `<|unk|>` (unknown) and `<|endoftext|>` (text separator) to our vocabulary.

In [ ]:
# Expand vocabulary list with special symbols
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer,token in enumerate(all_tokens)}

print("New Vocab size:", len(vocab.items()))

In [ ]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        # Fallback to <|unk|> if the word is not in the vocabulary
        preprocessed = [
            item if item in self.str_to_int 
            else "<|unk|>" for item in preprocessed
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Clean spacing around punctuation
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

In [ ]:
# Testing V2 tokenizer with multiple texts separated by <|endoftext|>
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = " <|endoftext|> ".join((text1, text2))

print(text)

In [ ]:
# Verify the ID of <|endoftext|>
print("ID of <|endoftext|>: ", vocab['<|endoftext|>'])

In [ ]:
# Encode
encoded_ids = tokenizer.encode(text)
print("Encoded:", encoded_ids)

In [ ]:
# Decode to inspect reconstruction. Unknown words Hello and palace become <|unk|>
print("Decoded:", tokenizer.decode(encoded_ids))

### 4.4 Visualizing Tokenization

Let's visualize the tokenized components using colors.

In [ ]:
from IPython.display import display, HTML

tokenizer = SimpleTokenizerV2(vocab)
token_ids = tokenizer.encode(text)
tokens = [tokenizer.decode([t]) for t in token_ids]

colors = [
    "#FFB3BA", "#FFDFBA", "#FFFFBA",
    "#BAFFC9", "#BAE1FF", "#D7BAFF"
]

html = ""
for i, token in enumerate(tokens):
    html += f"""
    <span style="
        background:{colors[i % len(colors)]};
        padding:4px 6px;
        margin:2px;
        color: #000;
        border-radius:4px;
        font-family:monospace;
    ">
    {repr(token)}
    </span>
    """

display(HTML(html))

## 5. BytePair Encoding (BPE)

Modern LLMs use BytePair Encoding (BPE) subword tokenizers. BPE merges frequent byte pairs to represent unrecognized words as combinations of subwords without needing an unknown tag.

### 5.1 Using Tiktoken for GPT-2 Tokenization

In [ ]:
import importlib
import tiktoken

print("tiktoken version:", importlib.metadata.version("tiktoken"))

In [ ]:
# Load the standard GPT-2 BPE tokenizer encoding
tokenizer = tiktoken.get_encoding("gpt2")

In [ ]:
# Encode and decode an unrecognized string sequence
e1 = tokenizer.encode('jkbvhgjv')
print("Encoded subword list:", e1)

d1 = tokenizer.decode(e1)
print("Decoded string:", d1)

In [ ]:
# Encode with special token preservation
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces "
     "of someunknownPlace."
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print("Tiktoken Encoded Integers:", integers)

In [ ]:
# Decode back
strings = tokenizer.decode(integers)
print("Tiktoken Decoded:", strings)

### 5.2 Tokenizing the Full Corpus with BPE

In [ ]:
# Load the text and tokenize it using the GPT-2 encoder
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print("Total Tiktoken Tokens:", len(enc_text))

In [ ]:
# Verify that decode reconstructs the original character count
decoded_len = len(tokenizer.decode(enc_text))
print("Reconstructed Character Count:", decoded_len)

## 6. Data Sampling with a Sliding Window

To train an autoregressive LLM, we chunk tokenized text into training sequences. The model predicts the next token given all prior tokens.

### 6.1 Creating Input-Target Pairs

In [ ]:
# Set sample encoding slice
enc_sample = enc_text[50:]

In [ ]:
context_size = 4

# Inputs (x) represent context window; targets (y) are shifted by 1 token
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(f"x (Inputs): {x}")
print(f"y (Targets):      {y}")

In [ ]:
# Visualize all predicting steps in the context
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(context, "---->", desired)

In [ ]:
# Decode variables to make it readable
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

### 6.2 Designing a PyTorch Dataset & DataLoader

Now let's build a PyTorch `Dataset` and `DataLoader` subclass to automatically chunk and batch the data.

In [ ]:
import torch
print("PyTorch version:", torch.__version__)

In [ ]:
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length, "Number of tokenized inputs must at least be equal to max_length+1"

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [ ]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [ ]:
# Demonstration of how python iterators work
# ex_iter = iter([1, 2, 3, 4])
# print(next(ex_iter))
# print(next(ex_iter))
# print(next(ex_iter))
# print(next(ex_iter))

In [ ]:
# Test the dataloader with a small batch_size and max_length
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=3, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print("First batch output:", first_batch)

input_batch, target_batch = first_batch
print("Decoded Input:", tokenizer.decode(input_batch[0].tolist()))

In [ ]:
# Fetch and print the next batch (shows overlapping stride mechanism)
second_batch = next(data_iter)
print("Second batch output:", second_batch)

input_batch, target_batch = second_batch
print("Decoded Input:", tokenizer.decode(input_batch[0].tolist()))
print("Decoded Test Array:", tokenizer.decode([2, 3, 5, 1]))

In [ ]:
# Inspecting how the subword tokenizer splits pre-spaced strings
print(tokenizer.encode(" learn"))

In [ ]:
# Create a larger batch representation
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
first_batch = next(data_iter)
input_batch, target_batch = first_batch

print("Inputs (Shape: [8, 4]):\n", input_batch)
print("Decoded input batch 0:", tokenizer.decode(input_batch[0].tolist()))

print("\nTargets (Shape: [8, 4]):\n", target_batch)
print("Decoded target batch 0:", tokenizer.decode(target_batch[0].tolist()))

## 7. Creating Token Embeddings

To represent tokens as continuous vector spaces, we use an embedding layer. Let's demonstrate this with a simple PyTorch `nn.Embedding` layer.

In [ ]:
input_ids = torch.tensor([2, 3, 5, 1])

# Vocabulary size is 6, embedding dimensions is 3
vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

print("Initial embedding layer weight matrix:")
print(embedding_layer.weight)

In [ ]:
# Querying the embedding layer weights for token index 3 and index 2
print("Embedding vector for ID 3:", embedding_layer(torch.tensor([3])))
print("Embedding vector for ID 2:", embedding_layer(torch.tensor([2])))

In [ ]:
# Embedding multiple token IDs at once
print("Embedding output shape:", embedding_layer(input_ids).shape)
print(embedding_layer(input_ids))

## 8. Encoding Word Positions (Positional Embeddings)

Self-attention mechanisms do not preserve positional sequence information. We must inject position embeddings to let the model learn the order of tokens in a sequence.

In [ ]:
# Setting parameters for token embedding layer
vocab_size = 50257
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
print(token_embedding_layer)

In [ ]:
# Print weight statistics and preview weights
print("Token embedding layer:", token_embedding_layer)
print("Weight tensor shape:", token_embedding_layer.weight.shape)
print(token_embedding_layer.weight)

In [ ]:
# Create a test dataloader batch to pass to the embedding layer
max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length,
    stride=max_length, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

In [ ]:
print("Token IDs:\n", inputs)
print("\nInputs shape:", inputs.shape)

In [ ]:
# Map token IDs to token embeddings
token_embeddings = token_embedding_layer(inputs)
print("Token embeddings shape:", token_embeddings.shape)
print(token_embeddings)

In [ ]:
# Create the absolute positional embedding layer matching context size (max_length)
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

print("Positional embedding weights:")
print(pos_embedding_layer.weight)

In [ ]:
# Embed positions 0 to max_length - 1
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print("Positional embeddings shape:", pos_embeddings.shape)
print(pos_embeddings)

In [ ]:
# Sum token embeddings and position embeddings to get the final representation
input_embeddings = token_embeddings + pos_embeddings
print("Final input embeddings shape:", input_embeddings.shape)
print(input_embeddings)